In [1]:
import polars as pl
import os
import gc

In [2]:
# ─── Cấu hình đường dẫn ──────────────────────────────────────────────
WORKING_DIR      = '/kaggle/working'
SENTIMENT_PATH   = '/kaggle/input/datasets/shinnraa/stage-34/review_sentiment_pog.parquet'
LEXICAL_PATH     = '/kaggle/input/datasets/shinnraa/stage-34/review_lexical.parquet'
 
# Output: 2 file riêng biệt thay vì 1
USER_FEATURES_PATH = os.path.join(WORKING_DIR, 'user_text_features.parquet')
ITEM_FEATURES_PATH = os.path.join(WORKING_DIR, 'item_text_features.parquet')

In [3]:
 # ════════════════════════════════════════════════════════════
# BƯỚC 0: KIỂM TRA FILE ĐẦU VÀO
# ════════════════════════════════════════════════════════════
for path, name in [(SENTIMENT_PATH, 'Sentiment'), (LEXICAL_PATH, 'Lexical')]:
    assert os.path.exists(path), f"❌ Không tìm thấy file {name}: {path}"
print("✅ Cả 2 file đầu vào đều tồn tại.\n")

✅ Cả 2 file đầu vào đều tồn tại.



In [4]:
# ════════════════════════════════════════════════════════════
# BƯỚC 1: LOAD DỮ LIỆU (1 LẦN DUY NHẤT)
# ════════════════════════════════════════════════════════════
print("1. Đang load dữ liệu...")
 
df_sent = pl.read_parquet(SENTIMENT_PATH)
df_lex  = pl.read_parquet(LEXICAL_PATH)
 
print(f"   Sentiment shape : {df_sent.shape}")
print(f"   Lexical shape   : {df_lex.shape}")
 
# Kiểm tra null nhanh
for col in ['prob_pos', 'prob_neu', 'prob_neg']:
    n = df_sent[col].null_count()
    if n > 0:
        print(f"   ⚠️  '{col}' có {n:,} null")

1. Đang load dữ liệu...
   Sentiment shape : (16509306, 6)
   Lexical shape   : (16509306, 34)


In [5]:
 # ════════════════════════════════════════════════════════════
# BƯỚC 2: TẠO USER PROFILE FEATURES
# ════════════════════════════════════════════════════════════
print("\n2. Đang tạo User Profile...")
 
# --- Nhánh Sentiment theo User ---
user_sent = (
    df_sent.group_by('mapped_user_id')
    .agg([
        # Loyalty: User này review nhiều không?
        pl.len().alias('user_total_reviews'),
 
        # Khẩu vị trung bình: User này dễ tính hay khó tính?
        pl.col('prob_pos').mean().alias('user_avg_prob_pos'),
        pl.col('prob_neu').mean().alias('user_avg_prob_neu'),
        pl.col('prob_neg').mean().alias('user_avg_prob_neg'),
 
        # Độ biến động: User này nhất quán hay thất thường?
        pl.col('prob_pos').std().fill_null(0.0).alias('user_std_prob_pos'),
        pl.col('prob_neg').std().fill_null(0.0).alias('user_std_prob_neg'),
 
        # Tín hiệu cực đoan: User đã từng rất hài lòng / rất phẫn nộ chưa?
        pl.col('prob_pos').max().alias('user_peak_satisfaction'),
        pl.col('prob_neg').max().alias('user_peak_anger'),
    ])
)
 
# --- Nhánh Lexical theo User ---
user_lex = (
    df_lex.group_by('mapped_user_id')
    .agg([
        # Văn phong: User này thường viết ngắn hay dài?
        pl.col('word_count').mean().alias('user_avg_word_count'),
        pl.col('word_count').max().cast(pl.Int32).alias('user_longest_review'),
        pl.col('word_count').std().fill_null(0.0).alias('user_std_word_count'),
 
        # Mức độ tâm huyết: Tỷ lệ review dài (> 50 từ)
        (
            (pl.col('word_count') > 50).sum().cast(pl.Float32) / pl.len()
        ).alias('user_ratio_long_reviews'),
    ])
)
 
# --- Merge User Profile ---
df_user_features = user_sent.join(user_lex, on='mapped_user_id', how='inner')
 
# Pre-join check
n_drop_user = len(user_sent) - len(df_user_features)
if n_drop_user > 0:
    print(f"   ⚠️  Mất {n_drop_user:,} users sau inner join (có trong sent nhưng không có trong lex)")
 
print(f"   ✅ User Profile shape: {df_user_features.shape}")
 
del user_sent, user_lex
gc.collect()


2. Đang tạo User Profile...
   ✅ User Profile shape: (2181749, 13)


0

In [6]:
# ════════════════════════════════════════════════════════════
# BƯỚC 3: TẠO ITEM PROFILE FEATURES
# ════════════════════════════════════════════════════════════
print("\n3. Đang tạo Item Profile...")
 
# --- Nhánh Sentiment theo Item ---
item_sent = (
    df_sent.group_by('mapped_item_id')
    .agg([
        # Độ phổ biến: Sản phẩm này được review nhiều không?
        pl.len().alias('item_total_reviews'),
 
        # Chất lượng cảm nhận chung từ đám đông
        pl.col('prob_pos').mean().alias('item_avg_prob_pos'),
        pl.col('prob_neu').mean().alias('item_avg_prob_neu'),
        pl.col('prob_neg').mean().alias('item_avg_prob_neg'),
 
        # Mức độ tranh cãi: Sản phẩm này gây phân cực không?
        # std cao → ý kiến trái chiều → tín hiệu riêng biệt với mean
        pl.col('prob_pos').std().fill_null(0.0).alias('item_controversy_score'),
 
        # Cờ rủi ro: Sản phẩm này đã từng bị chê cực đoan chưa?
        pl.col('prob_neg').max().alias('item_peak_risk_neg'),
        pl.col('prob_pos').max().alias('item_peak_satisfaction_pos'),
    ])
)
 
# --- Nhánh Lexical theo Item ---
item_lex = (
    df_lex.group_by('mapped_item_id')
    .agg([
        # Sản phẩm này có tạo ra nhiều tranh luận (review dài) không?
        pl.col('word_count').mean().alias('item_avg_word_count'),
        pl.col('word_count').sum().alias('item_total_words_received'),
 
        # Review dài nhất về sản phẩm này (tín hiệu cảm xúc cực đoan)
        pl.col('word_count').max().cast(pl.Int32).alias('item_longest_review_received'),
        # Tỷ lệ người mua viết review nghiêm túc (> 50 từ)
        (
            (pl.col('word_count') > 50).sum().cast(pl.Float32) / pl.len()
        ).alias('item_ratio_long_reviews'),
    ])
)
 
# --- Merge Item Profile ---
df_item_features = item_sent.join(item_lex, on='mapped_item_id', how='inner')
 
n_drop_item = len(item_sent) - len(df_item_features)
if n_drop_item > 0:
    print(f"   ⚠️  Mất {n_drop_item:,} items sau inner join")
 
print(f"   ✅ Item Profile shape: {df_item_features.shape}")
 
del item_sent, item_lex, df_sent, df_lex
gc.collect()


3. Đang tạo Item Profile...
   ✅ Item Profile shape: (565243, 12)


0

In [7]:
# ════════════════════════════════════════════════════════════
# BƯỚC 4: SANITY CHECK
# ════════════════════════════════════════════════════════════
print("\n4. Sanity check...")
 
for df, name in [(df_user_features, 'User'), (df_item_features, 'Item')]:
    # Kiểm tra null
    null_total = df.null_count().row(0)
    total_nulls = sum(null_total)
    if total_nulls > 0:
        print(f"   ⚠️  {name} Profile có {total_nulls:,} null!")
    else:
        print(f"   ✅ {name} Profile: không có null.")
 
    # Kiểm tra xác suất trong [0, 1]
    prob_cols = [c for c in df.columns if 'prob' in c or 'sentiment' in c
                 or 'satisfaction' in c or 'risk' in c or 'peak' in c]
    for col in prob_cols:
        mn, mx = df[col].min(), df[col].max()
        assert 0.0 <= mn and mx <= 1.0, \
            f"❌ {name}.{col} ngoài [0,1]: [{mn:.4f}, {mx:.4f}]"
 
    # Kiểm tra total_reviews >= 1
    review_col = 'user_total_reviews' if name == 'User' else 'item_total_reviews'
    assert df[review_col].min() >= 1, f"❌ {review_col} có giá trị < 1"
 
print("   ✅ Tất cả sanity checks PASS.")


4. Sanity check...
   ✅ User Profile: không có null.
   ✅ Item Profile: không có null.
   ✅ Tất cả sanity checks PASS.


In [8]:
# ════════════════════════════════════════════════════════════
# BƯỚC 5: XUẤT FILE
# ════════════════════════════════════════════════════════════
print("\n5. Đang ghi file...")
 
df_user_features.write_parquet(USER_FEATURES_PATH)
df_item_features.write_parquet(ITEM_FEATURES_PATH)
 
print(f"   📁 User features → {USER_FEATURES_PATH}")
print(f"   📁 Item features → {ITEM_FEATURES_PATH}")


5. Đang ghi file...
   📁 User features → /kaggle/working/user_text_features.parquet
   📁 Item features → /kaggle/working/item_text_features.parquet


In [9]:
import polars as pl
import os
import gc
import traceback

# ─── Cấu hình đường dẫn ──────────────────────────────────────────────
WORKING_DIR        = '/kaggle/working'
SENTIMENT_PATH     = '/kaggle/input/datasets/shinnraa/stage-34/review_sentiment_pog.parquet'
LEXICAL_PATH       = '/kaggle/input/datasets/shinnraa/stage-34/review_lexical.parquet'
USER_FEATURES_PATH = os.path.join(WORKING_DIR, 'user_text_features.parquet')
ITEM_FEATURES_PATH = os.path.join(WORKING_DIR, 'item_text_features.parquet')

# ─── Bộ đếm kết quả ──────────────────────────────────────────────────
PASS = 0
FAIL = 0

def check(name: str, condition: bool, detail: str = ""):
    global PASS, FAIL
    if condition:
        PASS += 1
        print(f"   ✅ PASS | {name}")
    else:
        FAIL += 1
        print(f"   ❌ FAIL | {name}" + (f" → {detail}" if detail else ""))

def section(title: str):
    print(f"\n{'═'*55}")
    print(f"  {title}")
    print(f"{'═'*55}")

# ════════════════════════════════════════════════════════════
# BLOCK 0: KIỂM TRA FILE TỒN TẠI
# ════════════════════════════════════════════════════════════
section("BLOCK 0: FILE TỒN TẠI")

for path, name in [
    (SENTIMENT_PATH,     'review_sentiment.parquet'),
    (LEXICAL_PATH,       'review_lexical.parquet'),
    (USER_FEATURES_PATH, 'user_text_features.parquet'),
    (ITEM_FEATURES_PATH, 'item_text_features.parquet'),
]:
    check(f"{name} tồn tại", os.path.exists(path), f"Không tìm thấy: {path}")

if FAIL > 0:
    print("\n🛑 Một hoặc nhiều file không tồn tại — dừng test.")
    raise SystemExit(1)

# ════════════════════════════════════════════════════════════
# BLOCK 1: LOAD DỮ LIỆU
# ════════════════════════════════════════════════════════════
section("BLOCK 1: LOAD DỮ LIỆU")

df_sent = pl.read_parquet(SENTIMENT_PATH)
df_lex  = pl.read_parquet(LEXICAL_PATH)
df_user = pl.read_parquet(USER_FEATURES_PATH)
df_item = pl.read_parquet(ITEM_FEATURES_PATH)

print(f"   Sentiment  : {df_sent.shape}")
print(f"   Lexical    : {df_lex.shape}")
print(f"   User Feat  : {df_user.shape}")
print(f"   Item Feat  : {df_item.shape}")

check("Sentiment có dữ liệu", df_sent.shape[0] > 0)
check("Lexical có dữ liệu",   df_lex.shape[0]  > 0)
check("User features có dữ liệu", df_user.shape[0] > 0)
check("Item features có dữ liệu", df_item.shape[0] > 0)

# ════════════════════════════════════════════════════════════
# BLOCK 2: KIỂM TRA SCHEMA (CỘT BẮT BUỘC)
# ════════════════════════════════════════════════════════════
section("BLOCK 2: SCHEMA — CỘT BẮT BUỘC")

REQUIRED_USER_COLS = [
    'mapped_user_id',
    'user_total_reviews',
    'user_avg_prob_pos', 'user_avg_prob_neu', 'user_avg_prob_neg',
    'user_std_prob_pos', 'user_std_prob_neg',
    'user_peak_satisfaction', 'user_peak_anger',
    'user_avg_word_count', 'user_longest_review',
    'user_std_word_count', 'user_ratio_long_reviews',
]

REQUIRED_ITEM_COLS = [
    'mapped_item_id',
    'item_total_reviews',
    'item_avg_prob_pos', 'item_avg_prob_neu', 'item_avg_prob_neg',
    'item_controversy_score',
    'item_peak_risk_neg', 'item_peak_satisfaction_pos',
    'item_avg_word_count', 'item_total_words_received',
    'item_longest_review_received', 'item_ratio_long_reviews',
]

for col in REQUIRED_USER_COLS:
    check(f"User có cột '{col}'", col in df_user.columns)

for col in REQUIRED_ITEM_COLS:
    check(f"Item có cột '{col}'", col in df_item.columns)

# ════════════════════════════════════════════════════════════
# BLOCK 3: KIỂM TRA NULL
# ════════════════════════════════════════════════════════════
section("BLOCK 3: NULL CHECK")

for df, name in [(df_user, 'User'), (df_item, 'Item')]:
    null_row = df.null_count().row(0)
    col_names = df.columns
    for col, n_null in zip(col_names, null_row):
        check(
            f"{name}.{col} không có null",
            n_null == 0,
            f"{n_null:,} null values"
        )

# ════════════════════════════════════════════════════════════
# BLOCK 4: KIỂM TRA GIÁ TRỊ HỢP LỆ
# ════════════════════════════════════════════════════════════
section("BLOCK 4: GIÁ TRỊ HỢP LỆ")

# --- Xác suất phải trong [0, 1] ---
prob_user_cols = [c for c in df_user.columns if any(
    kw in c for kw in ['prob', 'satisfaction', 'anger', 'ratio']
)]
prob_item_cols = [c for c in df_item.columns if any(
    kw in c for kw in ['prob', 'risk', 'satisfaction', 'controversy', 'ratio']
)]

for col in prob_user_cols:
    mn, mx = df_user[col].min(), df_user[col].max()
    check(f"User.{col} trong [0, 1]", 0.0 <= mn and mx <= 1.0,
          f"range=[{mn:.4f}, {mx:.4f}]")

for col in prob_item_cols:
    mn, mx = df_item[col].min(), df_item[col].max()
    check(f"Item.{col} trong [0, 1]", 0.0 <= mn and mx <= 1.0,
          f"range=[{mn:.4f}, {mx:.4f}]")

# --- total_reviews >= 1 ---
check("user_total_reviews >= 1",
      df_user['user_total_reviews'].min() >= 1,
      f"min={df_user['user_total_reviews'].min()}")

check("item_total_reviews >= 1",
      df_item['item_total_reviews'].min() >= 1,
      f"min={df_item['item_total_reviews'].min()}")

# --- word_count hợp lệ ---
check("user_avg_word_count > 0",
      df_user['user_avg_word_count'].min() > 0,
      f"min={df_user['user_avg_word_count'].min()}")

check("item_avg_word_count > 0",
      df_item['item_avg_word_count'].min() > 0,
      f"min={df_item['item_avg_word_count'].min()}")

# --- std >= 0 ---
for col in [c for c in df_user.columns if 'std' in c]:
    check(f"User.{col} >= 0", df_user[col].min() >= 0,
          f"min={df_user[col].min()}")

for col in [c for c in df_item.columns if 'std' in c or 'controversy' in c]:
    check(f"Item.{col} >= 0", df_item[col].min() >= 0,
          f"min={df_item[col].min()}")

# ════════════════════════════════════════════════════════════
# BLOCK 5: EDGE CASES
# ════════════════════════════════════════════════════════════
section("BLOCK 5: EDGE CASES")

# --- E1: User chỉ có 1 review → std phải = 0 ---
user_review_counts = (
    df_sent.group_by('mapped_user_id')
    .agg(pl.len().alias('cnt'))
)
single_review_users = user_review_counts.filter(pl.col('cnt') == 1)['mapped_user_id']

if len(single_review_users) > 0:
    sample_user = single_review_users[0]
    user_row = df_user.filter(pl.col('mapped_user_id') == sample_user)
    if len(user_row) > 0:
        std_val = user_row['user_std_prob_pos'][0]
        check(
            "User 1 review → user_std_prob_pos = 0.0",
            std_val == 0.0,
            f"std={std_val}"
        )
    else:
        check("User 1 review có trong User Profile", False,
              "user bị drop khỏi profile — kiểm tra inner join")
else:
    print("   ⚠️  Không có user nào chỉ có 1 review trong dataset này — bỏ qua E1")

# --- E2: Item chỉ có 1 review → controversy_score = 0 ---
item_review_counts = (
    df_sent.group_by('mapped_item_id')
    .agg(pl.len().alias('cnt'))
)
single_review_items = item_review_counts.filter(pl.col('cnt') == 1)['mapped_item_id']

if len(single_review_items) > 0:
    sample_item = single_review_items[0]
    item_row = df_item.filter(pl.col('mapped_item_id') == sample_item)
    if len(item_row) > 0:
        std_val = item_row['item_controversy_score'][0]
        check(
            "Item 1 review → item_controversy_score = 0.0",
            std_val == 0.0,
            f"controversy_score={std_val}"
        )
    else:
        check("Item 1 review có trong Item Profile", False,
              "item bị drop — kiểm tra inner join")
else:
    print("   ⚠️  Không có item nào chỉ có 1 review — bỏ qua E2")

# --- E3: User có trong Sentiment nhưng không có trong Lexical ---
users_only_in_sent = (
    df_sent.select('mapped_user_id').unique()
    .join(df_lex.select('mapped_user_id').unique(),
          on='mapped_user_id', how='anti')
)
n_missing = len(users_only_in_sent)
check(
    f"Users chỉ có trong Sentiment (bị drop sau inner join): {n_missing:,}",
    True  # Chỉ log thông tin, không fail
)
if n_missing > 0:
    loss_rate = n_missing / df_sent.select('mapped_user_id').n_unique()
    print(f"   ℹ️  Tỷ lệ mất: {loss_rate:.2%} — {'⚠️ Cần điều tra!' if loss_rate > 0.05 else 'OK'}")

# --- E4: Item có trong Sentiment nhưng không có trong Lexical ---
items_only_in_sent = (
    df_sent.select('mapped_item_id').unique()
    .join(df_lex.select('mapped_item_id').unique(),
          on='mapped_item_id', how='anti')
)
n_missing_item = len(items_only_in_sent)
check(
    f"Items chỉ có trong Sentiment (bị drop sau inner join): {n_missing_item:,}",
    True
)
if n_missing_item > 0:
    loss_rate_item = n_missing_item / df_sent.select('mapped_item_id').n_unique()
    print(f"   ℹ️  Tỷ lệ mất: {loss_rate_item:.2%} — {'⚠️ Cần điều tra!' if loss_rate_item > 0.05 else 'OK'}")

# --- E5: Không có user/item trùng lặp trong Profile ---
n_dup_user = len(df_user) - df_user.select('mapped_user_id').n_unique()
check("Không có user_id trùng lặp trong User Profile",
      n_dup_user == 0, f"{n_dup_user} duplicates")

n_dup_item = len(df_item) - df_item.select('mapped_item_id').n_unique()
check("Không có item_id trùng lặp trong Item Profile",
      n_dup_item == 0, f"{n_dup_item} duplicates")

# ════════════════════════════════════════════════════════════
# BLOCK 6: SIMULATE JOIN VỚI CANDIDATES (XGBoost readiness)
# ════════════════════════════════════════════════════════════
section("BLOCK 6: SIMULATE JOIN VỚI CANDIDATES")

# Lấy 1000 cặp (user, item) bất kỳ từ sentiment làm "candidates giả"
fake_candidates = df_sent.select(['mapped_user_id', 'mapped_item_id']).sample(
    n=min(1000, len(df_sent)), seed=42
).unique()

joined = (
    fake_candidates
    .join(df_user, on='mapped_user_id', how='left')
    .join(df_item, on='mapped_item_id', how='left')
)

# Tỷ lệ NULL sau join (candidates đã có lịch sử → phải gần 0%)
null_after_join = joined.null_count().to_numpy().sum()
total_cells = joined.shape[0] * (joined.shape[1] - 2)  # trừ 2 key cols
null_rate = null_after_join / total_cells if total_cells > 0 else 0

check(
    f"NULL rate sau left join vào candidates: {null_rate:.2%}",
    null_rate < 0.01,
    f"Quá nhiều NULL ({null_rate:.2%}) — kiểm tra inner join ở Stage 3"
)

check("Candidates sau join đủ số dòng",
      len(joined) == len(fake_candidates),
      f"Expected {len(fake_candidates)}, got {len(joined)}")

# ════════════════════════════════════════════════════════════
# KẾT QUẢ TỔNG HỢP
# ════════════════════════════════════════════════════════════
total = PASS + FAIL
print(f"""
╔══════════════════════════════════════════════════════╗
║              KẾT QUẢ TEST STAGE 3                   ║
╠══════════════════════════════════════════════════════╣
║  ✅ PASS : {PASS:<43}║
║  ❌ FAIL : {FAIL:<43}║
║  📊 TOTAL: {total:<43}║
╠══════════════════════════════════════════════════════╣
║  {'🎉 TẤT CẢ PASS — Stage 3 sẵn sàng cho XGBoost!' if FAIL == 0 else f'⚠️  CÓ {FAIL} TEST THẤT BẠI — Cần kiểm tra lại!':<52}║
╚══════════════════════════════════════════════════════╝
""")


═══════════════════════════════════════════════════════
  BLOCK 0: FILE TỒN TẠI
═══════════════════════════════════════════════════════
   ✅ PASS | review_sentiment.parquet tồn tại
   ✅ PASS | review_lexical.parquet tồn tại
   ✅ PASS | user_text_features.parquet tồn tại
   ✅ PASS | item_text_features.parquet tồn tại

═══════════════════════════════════════════════════════
  BLOCK 1: LOAD DỮ LIỆU
═══════════════════════════════════════════════════════
   Sentiment  : (16509306, 6)
   Lexical    : (16509306, 34)
   User Feat  : (2181749, 13)
   Item Feat  : (565243, 12)
   ✅ PASS | Sentiment có dữ liệu
   ✅ PASS | Lexical có dữ liệu
   ✅ PASS | User features có dữ liệu
   ✅ PASS | Item features có dữ liệu

═══════════════════════════════════════════════════════
  BLOCK 2: SCHEMA — CỘT BẮT BUỘC
═══════════════════════════════════════════════════════
   ✅ PASS | User có cột 'mapped_user_id'
   ✅ PASS | User có cột 'user_total_reviews'
   ✅ PASS | User có cột 'user_avg_prob_pos'
   ✅ PASS 

In [10]:
# ════════════════════════════════════════════════════════════
# BƯỚC 6 (BỔ SUNG): ĐIỀU TRA NGUYÊN NHÂN TRÙNG LẶP DỮ LIỆU
# ════════════════════════════════════════════════════════════
print("Đang tải lại dữ liệu Sentiment để điều tra trùng lặp...")

# Tải lại file Sentiment (chứa rating và xác suất cảm xúc)
df_investigate = pl.read_parquet(SENTIMENT_PATH)

# 1. Gom nhóm để tìm các cặp (user, item) bị lặp
duplicates_grouped = (
    df_investigate.group_by(['mapped_user_id', 'mapped_item_id'])
    .agg(pl.len().alias('count'))
    .filter(pl.col('count') > 1)
)
num_dup_pairs = duplicates_grouped.height
print(f"🔹 Số lượng cặp User-Item xuất hiện > 1 lần: {num_dup_pairs:,}")

if num_dup_pairs > 0:
    # 2. Kiểm tra lỗi nhân bản 100% (Giống hệt user, item, rating, và các cột xác suất)
    exact_duplicates = df_investigate.height - df_investigate.unique().height
    print(f"🔹 Số dòng bị trùng lặp CHÍNH XÁC 100% ở mọi cột: {exact_duplicates:,}")
    
    # 3. Spot-check những dòng trùng nhưng KHÁC rating (Kịch bản 2 & 3)
    suspects_df = df_investigate.join(duplicates_grouped, on=['mapped_user_id', 'mapped_item_id'], how='inner')
    
    different_values_dups = (
        suspects_df.group_by(['mapped_user_id', 'mapped_item_id'])
        .agg([
            pl.col('rating').n_unique().alias('unique_ratings'),
            pl.col('rating').alias('all_ratings'),
            pl.col('prob_pos').alias('all_prob_pos')
        ])
        .filter(pl.col('unique_ratings') > 1) # Lọc các ca có rating mâu thuẫn nhau
    )
    
    diff_count = different_values_dups.height
    print(f"🔹 Số cặp trùng nhưng CÓ RATING KHÁC NHAU (nghi ngờ update review / mua biến thể): {diff_count:,}")
    
    if diff_count > 0:
        print("\n--- 5 TRƯỜNG HỢP CÙNG USER, CÙNG ITEM NHƯNG KHÁC RATING ---")
        for row in different_values_dups.head(5).iter_rows(named=True):
            print(f"User: {row['mapped_user_id']} | Item: {row['mapped_item_id']}")
            print(f"  > Lịch sử Rating  : {row['all_ratings']}")
            print(f"  > Độ tích cực (Pos): {[round(p, 4) for p in row['all_prob_pos']]}")
            print("-" * 40)
    else:
        print("\n✅ Không có trường hợp nào khác rating. Cả cụm trùng lặp này đều giống nhau y hệt!")
        
    # --- ĐƯA RA KẾT LUẬN ---
    print("\n" + "="*50)
    print(" 💡 KẾT LUẬN & HƯỚNG XỬ LÝ ")
    print("="*50)
    if exact_duplicates > 0 and diff_count == 0:
        print("🚨 LỖI NHÂN BẢN HỆ THỐNG: Dữ liệu bị duplicate trong quá trình cào/join data trước đó.")
        print("👉 Hướng giải quyết: Quay lại BƯỚC 1, thêm lệnh sau vào dưới dòng đọc file:")
        print("   df_sent = df_sent.unique(subset=['mapped_user_id', 'mapped_item_id'])")
        print("   df_lex  = df_lex.unique(subset=['mapped_user_id', 'mapped_item_id'])")
    elif diff_count > 0:
        print("⚠️ CÓ BIẾN THỂ / UPDATE: Khách hàng viết nhiều review khác nhau cho cùng 1 sản phẩm.")
        print("👉 Hướng giải quyết: Nên sắp xếp và giữ lại dòng có cảm xúc rõ rệt nhất, hoặc dòng dài nhất.")
else:
    print("✅ Dữ liệu hoàn toàn sạch, không có cặp User-Item nào bị trùng!")

# Giải phóng RAM
del df_investigate, duplicates_grouped
if 'suspects_df' in locals(): del suspects_df
if 'different_values_dups' in locals(): del different_values_dups
gc.collect()

Đang tải lại dữ liệu Sentiment để điều tra trùng lặp...
🔹 Số lượng cặp User-Item xuất hiện > 1 lần: 167,892
🔹 Số dòng bị trùng lặp CHÍNH XÁC 100% ở mọi cột: 114,730
🔹 Số cặp trùng nhưng CÓ RATING KHÁC NHAU (nghi ngờ update review / mua biến thể): 15,510

--- 5 TRƯỜNG HỢP CÙNG USER, CÙNG ITEM NHƯNG KHÁC RATING ---
User: 1745642 | Item: 368881
  > Lịch sử Rating  : [4.0, 3.0]
  > Độ tích cực (Pos): [0.7178, 0.7974]
----------------------------------------
User: 683954 | Item: 560036
  > Lịch sử Rating  : [5.0, 4.0]
  > Độ tích cực (Pos): [0.9517, 0.0804]
----------------------------------------
User: 508822 | Item: 540111
  > Lịch sử Rating  : [4.0, 5.0]
  > Độ tích cực (Pos): [0.1187, 0.1863]
----------------------------------------
User: 526620 | Item: 357630
  > Lịch sử Rating  : [4.0, 4.0, 2.0]
  > Độ tích cực (Pos): [0.9316, 0.9546, 0.007]
----------------------------------------
User: 509448 | Item: 157692
  > Lịch sử Rating  : [4.0, 5.0]
  > Độ tích cực (Pos): [0.9663, 0.9526]
---

0

In [11]:
import polars as pl
import numpy as np

def run_sanity_check(df: pl.DataFrame, id_col: str, name: str):
    print(f"\n{'='*40}")
    print(f"🔍 KIỂM TRA BẢNG: {name.upper()}")
    print(f"{'='*40}")
    
    # 1. Kiểm tra trùng lặp ID
    total_rows = df.height
    unique_ids = df[id_col].n_unique()
    if total_rows != unique_ids:
        print(f"❌ LỖI NGHIÊM TRỌNG: Bảng có {total_rows} dòng nhưng chỉ có {unique_ids} ID độc lập. Bị trùng lặp ID!")
    else:
        print(f"✅ Pass: Khóa chính '{id_col}' duy nhất (Tổng: {total_rows:,} dòng).")

    # 2. Kiểm tra Null và Kiểu dữ liệu
    for col in df.columns:
        if col == id_col:
            continue
            
        # Kiểm tra Data Type (Phải là số)
        dtype = df[col].dtype
        if dtype not in [pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.UInt32]:
            print(f"❌ LỖI KIỂU DỮ LIỆU: Cột '{col}' đang ở dạng {dtype}. XGBoost chỉ nhận dạng số!")
            
        # Kiểm tra Null
        null_count = df[col].null_count()
        if null_count > 0:
            print(f"   ⚠️ Cảnh báo: Cột '{col}' có {null_count:,} giá trị Null ({(null_count/total_rows)*100:.2f}%).")

    # 3. Kiểm tra Infinity (Vô cực)
    numeric_cols = df.select(pl.selectors.numeric()).columns
    for col in numeric_cols:
        inf_count = df.filter(pl.col(col).is_infinite()).height
        if inf_count > 0:
            print(f"❌ LỖI NGHIÊM TRỌNG: Cột '{col}' chứa {inf_count} giá trị Infinity (Inf). XGBoost sẽ bị sập!")

    # 4. Kiểm tra Miền giá trị (Xác suất phải thuộc [0, 1], std >= 0)
    for col in numeric_cols:
        min_val = df[col].min()
        max_val = df[col].max()
        
        if 'prob' in col or 'satisfaction' in col or 'risk' in col or 'anger' in col:
            if min_val < 0.0 or max_val > 1.0:
                print(f"❌ LỖI TOÁN HỌC: Cột xác suất '{col}' nằm ngoài khoảng [0, 1]! (Min: {min_val}, Max: {max_val})")
                
        if 'std' in col or 'count' in col or 'total' in col:
            if min_val < 0:
                print(f"❌ LỖI TOÁN HỌC: Cột đếm/độ lệch chuẩn '{col}' có giá trị âm! (Min: {min_val})")
                
    print(f"🎉 Hoàn tất kiểm tra {name}.")

# --- CHẠY KIỂM TRA ---
user_path = '/kaggle/working/user_text_features.parquet'
item_path = '/kaggle/working/item_text_features.parquet'

df_user = pl.read_parquet(user_path)
df_item = pl.read_parquet(item_path)

run_sanity_check(df_user, id_col='mapped_user_id', name='User Features')
run_sanity_check(df_item, id_col='mapped_item_id', name='Item Features')


🔍 KIỂM TRA BẢNG: USER FEATURES
✅ Pass: Khóa chính 'mapped_user_id' duy nhất (Tổng: 2,181,749 dòng).
❌ LỖI KIỂU DỮ LIỆU: Cột 'user_longest_review' đang ở dạng UInt16. XGBoost chỉ nhận dạng số!
🎉 Hoàn tất kiểm tra User Features.

🔍 KIỂM TRA BẢNG: ITEM FEATURES
✅ Pass: Khóa chính 'mapped_item_id' duy nhất (Tổng: 565,243 dòng).
❌ LỖI KIỂU DỮ LIỆU: Cột 'item_longest_review_received' đang ở dạng UInt16. XGBoost chỉ nhận dạng số!
🎉 Hoàn tất kiểm tra Item Features.
